# LangGraph 심화 실습 — 정답

각 문제의 정답과 해설입니다.

## 0. 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.4")
print("\u2713 \ubaa8\ub378 \uc900\ube44 \uc644\ub8cc")

✓ 모델 준비 완료


---
## 1단계: 상태 리듀서

**핵심 개념**: `Annotated[list[str], operator.add]`를 쓰면 노드가 반환하는 리스트가 기존 리스트에 **누적(append)**됩니다. 리듀서가 없는 필드는 단순 **덮어쓰기(override)**됩니다.

- **Q1**: `Annotated` — typing 모듈에서 타입에 메타데이터를 붙이는 데 사용
- **Q2**: `operator` — `operator.add`는 리스트 + 리스트 연결 함수
- **Q3**: `Annotated[list[str], operator.add]` — items 필드에 리듀서 적용

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict  # Q1. Annotated를 import
import operator                           # Q2. operator 모듈 import

# Q3. items는 리스트 누적(operator.add), count는 단순 덮어쓰기
class LogState(TypedDict):
    items: Annotated[list[str], operator.add]  # 리듀서: 리스트 누적
    count: int                                  # 리듀서 없음: 덮어쓰기

def step_login(state: LogState) -> dict:
    return {"items": ["login"], "count": 1}

def step_search(state: LogState) -> dict:
    return {"items": ["search"], "count": 2}

def step_logout(state: LogState) -> dict:
    return {"items": ["logout"], "count": 3}

builder = StateGraph(LogState)
builder.add_node("login", step_login)
builder.add_node("search", step_search)
builder.add_node("logout", step_logout)

builder.add_edge(START, "login")
builder.add_edge("login", "search")
builder.add_edge("search", "logout")
builder.add_edge("logout", END)

graph = builder.compile()
result = graph.invoke({"items": [], "count": 0})

# items는 operator.add 리듀서로 누적: ['login'] + ['search'] + ['logout']
# count는 리듀서 없이 덮어쓰기: 1 → 2 → 3 (최종 3)
print(f"items: {result['items']}")  # ['login', 'search', 'logout']
print(f"count: {result['count']}")  # 3

items: ['login', 'search', 'logout']
count: 3


---
## 2단계: 조건부 엣지

**핵심 개념**: `add_conditional_edges(source, routing_fn, mapping)`을 사용하면 라우팅 함수의 반환값에 따라 다른 노드로 분기합니다.

- **Q4**: `state["label"]` — check_number 노드가 설정한 label 값을 그대로 반환
- **Q5**: `add_conditional_edges("check", route_number, {...})` — 조건부 엣지 등록
- **Q6**: `END` — 각 처리 노드에서 그래프 종료로 연결

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class NumberState(TypedDict):
    number: int
    label: str
    result: str

def check_number(state: NumberState) -> dict:
    n = state["number"]
    if n > 0:
        return {"label": "positive"}
    elif n < 0:
        return {"label": "negative"}
    return {"label": "zero"}

def handle_positive(state: NumberState) -> dict:
    return {"result": f"{state['number']}\uc740 \uc591\uc218\uc785\ub2c8\ub2e4."}

def handle_negative(state: NumberState) -> dict:
    return {"result": f"{state['number']}\uc740 \uc74c\uc218\uc785\ub2c8\ub2e4."}

def handle_zero(state: NumberState) -> dict:
    return {"result": "0\uc785\ub2c8\ub2e4."}

# Q4. 라우팅 함수: state의 label 값을 반환
def route_number(state: NumberState) -> str:
    return state["label"]  # "positive", "negative", "zero" 중 하나

builder = StateGraph(NumberState)
builder.add_node("check", check_number)
builder.add_node("positive", handle_positive)
builder.add_node("negative", handle_negative)
builder.add_node("zero", handle_zero)

builder.add_edge(START, "check")

# Q5. add_conditional_edges: 소스 노드, 라우팅 함수, 매핑 딕셔너리
builder.add_conditional_edges("check", route_number, {
    "positive": "positive",
    "negative": "negative",
    "zero": "zero",
})

# Q6. 각 처리 노드 → END로 연결
builder.add_edge("positive", END)
builder.add_edge("negative", END)
builder.add_edge("zero", END)

graph = builder.compile()

for num in [42, -7, 0]:
    result = graph.invoke({"number": num})
    print(result["result"])

42은 양수입니다.
-7은 음수입니다.
0입니다.


---
## 3단계: MessagesState + LLM 노드

**핵심 개념**: `MessagesState`는 내부에 `messages: Annotated[list, add_messages]`가 정의된 사전 제공 상태입니다. 메시지를 반환하면 `add_messages` 리듀서가 자동으로 누적합니다.

- **Q7**: `MessagesState` — langgraph.graph에서 import
- **Q8**: `SystemMessage` — 시스템 역할 지정용 메시지
- **Q9**: `model.invoke(state["messages"])` 호출 후 응답(response)을 리스트에 넣어 반환
- **Q10**: `SystemMessage(...)`, `HumanMessage(...)` — 각각의 메시지 클래스로 생성

In [4]:
from langgraph.graph import MessagesState, START, END  # Q7. MessagesState import
from langgraph.graph import StateGraph
from langchain.messages import HumanMessage, SystemMessage  # Q8. SystemMessage import

# Q9. chatbot 노드
def chatbot(state: MessagesState) -> dict:
    response = model.invoke(state["messages"])  # state의 messages를 LLM에 전달
    return {"messages": [response]}              # 응답을 messages에 추가

builder = StateGraph(MessagesState)  # MessagesState로 그래프 생성
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

# Q10. SystemMessage + HumanMessage로 호출
result = graph.invoke({
    "messages": [
        SystemMessage(content="\ub2f9\uc2e0\uc740 \uc218\ud559 \uc120\uc0dd\ub2d8\uc785\ub2c8\ub2e4."),
        HumanMessage(content="\ud53c\ud0c0\uace0\ub77c\uc2a4 \uc815\ub9ac\ub97c \ud55c \ubb38\uc7a5\uc73c\ub85c \uc124\uba85\ud574\uc8fc\uc138\uc694."),
    ]
})

print("\uc751\ub2f5:", result["messages"][-1].content)

응답: 직각삼각형에서 빗변의 제곱은 나머지 두 변의 제곱의 합과 같습니다.


---
## 4단계: 입출력 스키마

**핵심 개념**: `StateGraph(InternalState, input_schema=..., output_schema=...)`으로 내부 상태와 외부 인터페이스를 분리합니다. 중간 처리용 필드(intermediate)는 외부에 노출되지 않습니다.

- **Q11**: `question: str` — 외부에서 받는 입력 필드
- **Q12**: `answer: str` — 외부로 내보내는 출력 필드
- **Q13**: `StateGraph(InternalState, input_schema=InputSchema, output_schema=OutputSchema)`

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Q11. 입력 스키마: question만 받음
class InputSchema(TypedDict):
    question: str

# Q12. 출력 스키마: answer만 내보냄
class OutputSchema(TypedDict):
    answer: str

# 내부 상태: 중간 처리 필드 포함
class InternalState(TypedDict):
    question: str
    answer: str
    intermediate: str  # 외부에 노출되지 않음

def preprocess(state: InternalState) -> dict:
    return {"intermediate": state["question"].upper()}

def generate_answer(state: InternalState) -> dict:
    return {"answer": f"'{state['intermediate']}'\uc5d0 \ub300\ud55c \ub2f5\ubcc0\uc785\ub2c8\ub2e4."}

# Q13. 내부 상태, 입력 스키마, 출력 스키마를 전달
builder = StateGraph(
    InternalState,
    input_schema=InputSchema,
    output_schema=OutputSchema,
)

builder.add_node("preprocess", preprocess)
builder.add_node("answer", generate_answer)

builder.add_edge(START, "preprocess")
builder.add_edge("preprocess", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

# 입력은 question만, 출력은 answer만 (intermediate는 숨겨짐)
result = graph.invoke({"question": "\ub7ad\uadf8\ub798\ud504\ub780?"})
print(result)  # {'answer': "'랭그래프란?'에 대한 답변입니다."}

{'answer': "'랭그래프란?'에 대한 답변입니다."}


---
## 5단계: Prompt Chaining 패턴

**핵심 개념**: 첫 번째 노드의 출력(draft)이 두 번째 노드의 입력이 되는 순차 체인입니다. 각 노드에서 `model.invoke()`를 호출하고 `response.content`로 텍스트를 꺼냅니다.

- **Q14**: `model.invoke(...)` 호출, `state['topic']`으로 주제 참조, `response.content`로 결과 추출
- **Q15**: `state['draft']`로 초안 참조, `response.content`로 결과 추출
- **Q16**: 노드 이름 `"draft"`, `"improve"` 등록, `START → draft → improve → END` 엣지 연결

In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class ChainState(TypedDict):
    topic: str
    draft: str
    improved: str

# Q14. topic을 받아 한 문장 설명을 생성
def generate_draft(state: ChainState) -> dict:
    response = model.invoke(f"\ub2e4\uc74c\uc5d0 \ub300\ud574 \ud55c \ubb38\uc7a5\uc73c\ub85c \uc124\uba85\ud574\uc8fc\uc138\uc694: {state['topic']}")
    return {"draft": response.content}

# Q15. draft를 받아 개선
def improve_draft(state: ChainState) -> dict:
    response = model.invoke(f"\ub2e4\uc74c \ubb38\uc7a5\uc744 \ub354 \ub9e4\ub825\uc801\uc73c\ub85c \uac1c\uc120\ud574\uc8fc\uc138\uc694: {state['draft']}")
    return {"improved": response.content}

builder = StateGraph(ChainState)

# Q16. 노드 등록 및 엣지 연결
builder.add_node("draft", generate_draft)
builder.add_node("improve", improve_draft)

builder.add_edge(START, "draft")      # START → draft
builder.add_edge("draft", "improve")  # draft → improve
builder.add_edge("improve", END)      # improve → END

chain = builder.compile()
result = chain.invoke({"topic": "\uc778\uacf5\uc9c0\ub2a5"})
print(f"\ucd08\uc548: {result['draft']}")
print(f"\uac1c\uc120: {result['improved']}")

초안: 인공지능은 컴퓨터가 인간의 학습, 추론, 판단 같은 지적 능력을 모방해 문제를 해결하도록 만드는 기술입니다.
개선: 물론입니다. 더 매력적으로 다듬으면 이렇게 표현할 수 있습니다:

**인공지능은 컴퓨터가 인간의 학습, 추론, 판단과 같은 지적 능력을 모방해 스스로 문제를 이해하고 해결하도록 하는 혁신적인 기술입니다.**

원하시면 제가 이 문장을  
- **더 전문적으로**
- **더 쉽게**
- **더 광고/홍보 문구처럼**
- **더 세련되고 임팩트 있게**  

바꿔드릴 수도 있습니다.


---
## 6단계: Parallelization 패턴

**핵심 개념**: START에서 여러 노드로 엣지를 연결하면 **병렬 실행**됩니다. `Annotated[list, operator.add]` 리듀서가 있어야 두 노드의 결과가 하나의 리스트로 합쳐집니다.

- **Q17**: `Annotated[list[str], operator.add]` — 병렬 노드 결과를 누적
- **Q18**: `START`에서 두 노드로 각각 엣지 연결 (이것이 병렬 분기)
- **Q19**: 두 노드에서 synthesize로 합류, synthesize에서 `END`로

In [7]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
import operator

# Q17. analyses를 리스트 누적 리듀서로 정의
class ParallelState(TypedDict):
    text: str
    analyses: Annotated[list[str], operator.add]

def analyze_sentiment(state: ParallelState) -> dict:
    r = model.invoke(f"\ud55c \ubb38\uc7a5\uc73c\ub85c \uac10\uc815\uc744 \ubd84\uc11d\ud574\uc8fc\uc138\uc694: {state['text']}")
    return {"analyses": [f"\uac10\uc815: {r.content}"]}

def extract_keywords(state: ParallelState) -> dict:
    r = model.invoke(f"\ud575\uc2ec \ud0a4\uc6cc\ub4dc 3\uac1c\ub97c \ucd94\ucd9c\ud574\uc8fc\uc138\uc694: {state['text']}")
    return {"analyses": [f"\ud0a4\uc6cc\ub4dc: {r.content}"]}

def synthesize(state: ParallelState) -> dict:
    return {"analyses": [f"\uc885\ud569: {len(state['analyses'])}\uac1c \ubd84\uc11d \uc644\ub8cc"]}

builder = StateGraph(ParallelState)
builder.add_node("sentiment", analyze_sentiment)
builder.add_node("keywords", extract_keywords)
builder.add_node("synthesize", synthesize)

# Q18. START에서 두 노드로 병렬 분기
builder.add_edge(START, "sentiment")
builder.add_edge(START, "keywords")

# Q19. 두 노드 → synthesize → END
builder.add_edge("sentiment", "synthesize")
builder.add_edge("keywords", "synthesize")

builder.add_edge("synthesize", END)

parallel_graph = builder.compile()
result = parallel_graph.invoke({"text": "AI \uae30\uc220\uc774 \uc758\ub8cc \ubd84\uc57c\ub97c \ud601\uc2e0\ud558\uace0 \uc788\ub2e4.", "analyses": []})

for a in result["analyses"]:
    print(f"  {a}")

  키워드: - AI 기술
- 의료 분야
- 혁신
  감정: 이 문장은 **긍정적인 감정**을 담고 있으며, AI 기술이 의료 분야에 **혁신적이고 유익한 변화를 가져오고 있다**는 기대와 낙관의 뉘앙스를 전달합니다.
  종합: 2개 분석 완료


---
## 7단계: Routing 패턴

**핵심 개념**: `model.with_structured_output(PydanticModel)`을 사용하면 LLM이 정해진 스키마(Literal 값)로만 응답합니다. 그 결과를 라우팅 함수에서 읽어 분기합니다.

- **Q20**: `Literal["science", "history", "culture"]` — 허용 카테고리 정의
- **Q21**: `model.with_structured_output(Classification)` 호출, `result.category`로 접근
- **Q22**: `state["category"]` 반환
- **Q23**: `add_conditional_edges("classify", route, {...})`

In [8]:
from pydantic import BaseModel
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END

# Q20. 분류 카테고리를 Literal로 정의
class Classification(BaseModel):
    category: Literal["science", "history", "culture"]

class RouteState(TypedDict):
    question: str
    category: str
    answer: str

# Q21. LLM structured output으로 분류
def classify(state: RouteState) -> dict:
    structured = model.with_structured_output(Classification)  # Pydantic 모델 바인딩
    result = structured.invoke(f"\ub2e4\uc74c \uc9c8\ubb38\uc744 science/history/culture \uc911 \ud558\ub098\ub85c \ubd84\ub958\ud558\uc138\uc694: {state['question']}")
    return {"category": result.category}  # Pydantic 모델의 속성으로 접근

def handle_science(state: RouteState) -> dict:
    r = model.invoke(f"\uacfc\ud559 \uc804\ubb38\uac00\ub85c\uc11c \ub2f5\ubcc0: {state['question']}")
    return {"answer": r.content}

def handle_history(state: RouteState) -> dict:
    r = model.invoke(f"\uc5ed\uc0ac \uc804\ubb38\uac00\ub85c\uc11c \ub2f5\ubcc0: {state['question']}")
    return {"answer": r.content}

def handle_culture(state: RouteState) -> dict:
    r = model.invoke(f"\ubb38\ud654 \uc804\ubb38\uac00\ub85c\uc11c \ub2f5\ubcc0: {state['question']}")
    return {"answer": r.content}

# Q22. 라우팅 함수: category 값을 반환
def route(state: RouteState) -> str:
    return state["category"]

builder = StateGraph(RouteState)
builder.add_node("classify", classify)
builder.add_node("science", handle_science)
builder.add_node("history", handle_history)
builder.add_node("culture", handle_culture)

builder.add_edge(START, "classify")

# Q23. 조건부 엣지
builder.add_conditional_edges(
    "classify",
    route,
    {
        "science": "science",
        "history": "history",
        "culture": "culture",
    }
)

builder.add_edge("science", END)
builder.add_edge("history", END)
builder.add_edge("culture", END)

router = builder.compile()

result = router.invoke({"question": "\ube57\uc758 \uc18d\ub3c4\ub294 \uc5bc\ub9c8\uc778\uac00\uc694?"})
print(f"\uce74\ud14c\uace0\ub9ac: {result['category']}")
print(f"\ub2f5\ubcc0: {result['answer'][:200]}")

카테고리: science
답변: 질문하신 **“빗의 속도”**는 보통 **빗방울이 떨어지는 속도**를 뜻하는 것으로 이해할 수 있습니다.

## 짧게 답하면
빗방울의 속도는 보통 **초당 2~9미터(m/s)** 정도입니다.  
즉, 시속으로 바꾸면 대략 **7~32km/h** 정도입니다.

## 왜 속도가 다를까?
빗방울은 크기에 따라 속도가 달라집니다.

- **아주 작은 빗방울**: 


---
## 8단계: Orchestrator-Worker 패턴

**핵심 개념**: `Send(node_name, state_dict)`를 사용하면 런타임에 **동적으로 워커 노드를 여러 개 생성**할 수 있습니다. 각 Send는 독립적인 WorkerState를 가지고 병렬 실행됩니다.

- **Q24**: `Send` — `langgraph.types`에서 import
- **Q25**: `Send("worker", {"section": s})` — 각 섹션을 worker 노드에 분배
- **Q26**: `add_conditional_edges("plan", assign_workers, ["worker"])` — 동적 분기

In [9]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send  # Q24. Send import
from typing import Annotated, TypedDict
import operator

class OrchestratorState(TypedDict):
    topic: str
    sections: list[str]
    results: Annotated[list[str], operator.add]

class WorkerState(TypedDict):
    section: str

def plan_sections(state: OrchestratorState) -> dict:
    r = model.invoke(
        f"'{state['topic']}'\uc5d0 \ub300\ud55c \uc9e7\uc740 \uc139\uc158 \uc81c\ubaa9 3\uac1c\ub97c \ub098\uc5f4\ud574\uc8fc\uc138\uc694. \ud55c \uc904\uc5d0 \ud558\ub098\uc529, \ubc88\ud638 \uc5c6\uc774."
    )
    sections = [s.strip() for s in r.content.strip().split("\n") if s.strip()][:3]
    return {"sections": sections}

# Q25. Send()로 각 섹션을 worker에 분배
def assign_workers(state: OrchestratorState) -> list[Send]:
    return [
        Send("worker", {"section": s})  # 각 섹션마다 worker 인스턴스 생성
        for s in state["sections"]
    ]

def worker(state: WorkerState) -> dict:
    r = model.invoke(f"\ub2e4\uc74c\uc5d0 \ub300\ud574 \ud55c \ubb38\uc7a5\uc73c\ub85c \uc791\uc131\ud574\uc8fc\uc138\uc694: {state['section']}")
    return {"results": [f"## {state['section']}\n{r.content}"]}

builder = StateGraph(OrchestratorState)
builder.add_node("plan", plan_sections)
builder.add_node("worker", worker)

builder.add_edge(START, "plan")

# Q26. plan → assign_workers로 조건부 엣지 (동적 Send)
builder.add_conditional_edges("plan", assign_workers, ["worker"])

builder.add_edge("worker", END)

orchestrator = builder.compile()
result = orchestrator.invoke({"topic": "\ud074\ub77c\uc6b0\ub4dc \ucef4\ud4e8\ud305", "sections": [], "results": []})

for r in result["results"]:
    print(r)
    print()

## 클라우드 컴퓨팅의 개요
클라우드 컴퓨팅은 인터넷을 통해 서버, 저장소, 데이터베이스, 네트워크, 소프트웨어 등 다양한 IT 자원을 필요할 때마다 유연하게 제공하고 사용량에 따라 비용을 지불하는 컴퓨팅 서비스 방식입니다.

## 주요 서비스 모델
주요 서비스 모델은 고객의 다양한 요구를 충족하기 위해 핵심 기능과 가치를 중심으로 설계된 대표 서비스 제공 방식입니다.

## 도입 효과와 활용 사례
도입 효과로는 업무 효율성 향상, 비용 절감, 품질 개선, 의사결정 고도화 등이 있으며, 활용 사례로는 고객 상담 자동화, 데이터 분석 기반 수요 예측, 생산 공정 최적화, 맞춤형 마케팅 등이 있습니다.



---
## 9단계: Evaluator-Optimizer 패턴

**핵심 개념**: 조건부 엣지로 **루프**를 만들어, 품질이 충분할 때까지 생성→평가를 반복합니다. `should_retry` 함수가 `END` 또는 `"generate"`를 반환하여 분기합니다.

- **Q27**: `state.get("feedback")` — 피드백이 있으면 개선 모드
- **Q28**: `range(8, 11)` — 8, 9, 10점을 합격으로 판정
- **Q29**: `state["is_good"]`, `state["iterations"]` 확인, `END` 또는 `"generate"` 반환
- **Q30**: `add_conditional_edges("evaluate", should_retry, ["generate", END])`

In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class EvalState(TypedDict):
    task: str
    draft: str
    feedback: str
    is_good: bool
    iterations: int

# Q27. 피드백이 있으면 개선, 없으면 새로 생성
def generate(state: EvalState) -> dict:
    if state.get("feedback"):  # 피드백 존재 여부 확인
        prompt = f"\ud53c\ub4dc\ubc31\uc744 \ubc18\uc601\ud558\uc5ec \uac1c\uc120\ud574\uc8fc\uc138\uc694.\n\uc6d0\ubcf8: {state['draft']}\n\ud53c\ub4dc\ubc31: {state['feedback']}"
    else:
        prompt = f"\ub2e4\uc74c\uc5d0 \ub300\ud55c \ud55c \ubb38\uc7a5 \uc2ac\ub85c\uac74\uc744 \uc791\uc131\ud574\uc8fc\uc138\uc694: {state['task']}"
    r = model.invoke(prompt)
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

# Q28. 8~10점이면 합격
def evaluate(state: EvalState) -> dict:
    r = model.invoke(f"\uc774 \uc2ac\ub85c\uac74\uc744 1-10\uc73c\ub85c \ud3c9\uac00\ud558\uace0 \uac04\ub2e8\ud55c \ud53c\ub4dc\ubc31\uc744 \uc8fc\uc138\uc694: '{state['draft']}'")
    content = r.content
    is_good = any(f"{n}/10" in content for n in range(8, 11))  # 8/10, 9/10, 10/10
    return {"feedback": content, "is_good": is_good}

# Q29. 종료 조건: 합격이거나 3회 반복
def should_retry(state: EvalState) -> str:
    if state["is_good"] or state["iterations"] >= 3:
        return END       # 종료
    return "generate"    # 다시 생성

builder = StateGraph(EvalState)
builder.add_node("generate", generate)
builder.add_node("evaluate", evaluate)

builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")

# Q30. evaluate → should_retry로 조건부 엣지 (루프 또는 종료)
builder.add_conditional_edges("evaluate", should_retry, ["generate", END])

optimizer = builder.compile()
result = optimizer.invoke({"task": "\ud658\uacbd \ubcf4\ud638 \ucea0\ud398\uc778"})
print(f"\ucd5c\uc885 \uc2ac\ub85c\uac74 ({result['iterations']}\ubc88 \ubc18\ubcf5): {result['draft']}")

최종 슬로건 (1번 반복): 지구를 지키는 가장 쉬운 실천, 오늘의 작은 변화에서 시작됩니다.


---
## 전체 정답 요약

| 문제 | 정답 | 설명 |
|------|------|------|
| Q1 | `Annotated` | typing에서 타입+메타데이터를 합치는 도구 |
| Q2 | `operator` | `operator.add`로 리스트 누적 리듀서 정의 |
| Q3 | `Annotated[list[str], operator.add]` | 리듀서 적용 문법 |
| Q4 | `state["label"]` | 라우팅 함수는 분기 키를 반환 |
| Q5 | `add_conditional_edges("check", route_number, {...})` | 조건부 엣지 등록 |
| Q6 | `END` | 처리 노드에서 그래프 종료 |
| Q7 | `MessagesState` | LLM 대화용 사전 정의 상태 |
| Q8 | `SystemMessage` | 시스템 역할 메시지 |
| Q9 | `model.invoke(state["messages"])` → `[response]` | LLM 호출 후 응답 반환 |
| Q10 | `SystemMessage(...)`, `HumanMessage(...)` | 메시지 객체 생성 |
| Q11 | `question: str` | 입력 스키마 필드 |
| Q12 | `answer: str` | 출력 스키마 필드 |
| Q13 | `InternalState, InputSchema, OutputSchema` | StateGraph 인자 3개 |
| Q14 | `model.invoke(...)`, `state['topic']`, `response.content` | LLM 호출 패턴 |
| Q15 | `model.invoke(...)`, `state['draft']`, `response.content` | 체인 두 번째 노드 |
| Q16 | `"draft"`, `"improve"`, `START`, `END` | 노드 등록 및 엣지 연결 |
| Q17 | `Annotated[list[str], operator.add]` | 병렬 결과 누적 |
| Q18 | `START`, `START` | START에서 두 노드로 분기 = 병렬 |
| Q19 | `"sentiment"`, `"keywords"`, `END` | 합류 후 종료 |
| Q20 | `"science"`, `"history"`, `"culture"` | Literal 카테고리 정의 |
| Q21 | `with_structured_output`, `result.category` | 구조화 출력 |
| Q22 | `state["category"]` | 라우팅 함수 반환값 |
| Q23 | `add_conditional_edges`, `route` | 조건부 엣지 등록 |
| Q24 | `Send` | 동적 워커 생성용 import |
| Q25 | `Send("worker", {"section": s})` | 워커에 섹션 분배 |
| Q26 | `add_conditional_edges("plan", assign_workers, ["worker"])` | 동적 분기 |
| Q27 | `state.get("feedback")` | 피드백 존재 여부 확인 |
| Q28 | `range(8, 11)` | 8~10점 합격 판정 |
| Q29 | `is_good`, `iterations`, `END`, `"generate"` | 종료/반복 조건 |
| Q30 | `add_conditional_edges("evaluate", should_retry, [...])` | 루프 엣지 |